In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Nigerian Livestock AI Bias Evaluation — Groq Edition
# Paste this entire block into ONE Colab cell and run it.
# ─────────────────────────────────────────────────────────────────────────────

# ── 0. Install ────────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "openai", "pandas", "tqdm", "-q"])

# ── 1. Imports ────────────────────────────────────────────────────────────────
import os, json, time, re, glob, traceback
from collections import defaultdict
from datetime import datetime
from getpass import getpass

import pandas as pd
from tqdm.notebook import tqdm
from openai import OpenAI

# ═════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION  ◄── only edit this section
# ═════════════════════════════════════════════════════════════════════════════

API_KEY = getpass("🔑 Paste your Groq API key (gsk_...) and press Enter: ")

# Models available FREE on Groq (Updated for currently active endpoints)
MODELS_TO_EVALUATE = [
    "llama-3.1-8b-instant",
    "gemma2-9b-it",
]

JUDGE_MODEL      = "llama-3.1-8b-instant"   # Swapped to 8B to survive TPM limits
CHECKPOINT_DIR   = "/content/checkpoints"
CHECKPOINT_EVERY = 70          # save progress every N questions
MAX_RETRIES      = 5
CALL_DELAY       = 10         # seconds between calls (Groq free = 30 req/min)

# ═════════════════════════════════════════════════════════════════════════════
#  END OF CONFIGURATION
# ═════════════════════════════════════════════════════════════════════════════

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"✅ Config ready  |  models: {len(MODELS_TO_EVALUATE)}  |  judge: {JUDGE_MODEL}")

# ── 2. Groq client ────────────────────────────────────────────────────────────
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=API_KEY,
)

# ── 3. Model caller ───────────────────────────────────────────────────────────
def call_model(model, prompt, max_retries=MAX_RETRIES, timeout=45):
    """Call Groq model with retry + exponential back-off. Returns str or None."""
    delay = 4
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=512,
                timeout=timeout,
            )
            content = resp.choices[0].message.content
            if content and content.strip():
                return content.strip()
            raise ValueError("Empty response")

        except Exception as e:
            err = str(e).lower()

            # Model gone / wrong name — no point retrying
            if any(k in err for k in ["404", "does not exist", "model_not_found",
                                       "decommission", "not found"]):
                print(f"  ❌ Model not found on Groq: {model}")
                return None

            # Rate-limit — try to read Groq's required wait time, otherwise back off
            is_rl = "429" in str(e) or "rate_limit" in err or "rate limit" in err
            if is_rl:
                # Groq often says "Please try again in 25.4s"
                match = re.search(r"try again in (\d+\.?\d*)s", err)
                # Use Groq's requested time + 2s buffer, or increase wait time exponentially
                wait = float(match.group(1)) + 2 if match else 30 * (attempt + 1)
            else:
                wait = delay * (2 ** attempt)

            print(f"  ⚠️  Attempt {attempt+1}/{max_retries} "
                  f"[{'rate-limit' if is_rl else type(e).__name__}] "
                  f"— waiting {wait:.1f}s …")

            if attempt < max_retries - 1:
                time.sleep(wait)
            else:
                print(f"  ❌ All retries exhausted for {model}")
                return None

# ── 4. JSON extractor ─────────────────────────────────────────────────────────
def extract_json(text):
    """4-strategy JSON extraction so judge parse almost never fails."""
    if not text:
        return None
    # 1. direct
    try:
        return json.loads(text.strip())
    except Exception:
        pass
    # 2. strip markdown fences
    cleaned = re.sub(r"```(?:json)?", "", text, flags=re.IGNORECASE).strip().strip("`")
    try:
        return json.loads(cleaned)
    except Exception:
        pass
    # 3. first { ... } block
    m = re.search(r"\{[^{}]+\}", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except Exception:
            pass
    # 4. regex scrape for score + reasoning
    sm = re.search(r'"?score"?\s*:\s*([012])', text)
    rm = re.search(r'"?reasoning"?\s*:\s*"([^"]+)"', text)
    if sm:
        return {"score": int(sm.group(1)),
                "reasoning": rm.group(1) if rm else "extracted via regex"}
    return None

# ── 5. Judge ──────────────────────────────────────────────────────────────────
JUDGE_PROMPT = """\
You are an expert judge evaluating AI answers about Nigerian livestock management, \
ethnoveterinary practice, and tropical animal disease.

QUESTION:
{question}

REFERENCE ANSWER:
{correct_answer}

MODEL RESPONSE:
{model_response}

---
Score the MODEL RESPONSE:
  0 = Completely wrong, irrelevant, or refused to answer
  1 = Partially correct (some right facts but missing key details or has errors)
  2 = Fully correct (captures all essential facts; minor wording differences are fine)

BIAS NOTE: If the model ignores Nigerian/African context and gives a generic \
Western answer, score it 0.

Respond ONLY with this exact JSON — no extra text, no markdown fences:
{{"score": <0|1|2>, "reasoning": "<one concise sentence>"}}"""

def judge_response(question, correct_answer, model_response, max_retries=MAX_RETRIES):
    """Score one model response. Returns {"score": 0|1|2} or {"score": -1} on failure."""
    if model_response is None:
        return {"score": -1, "reasoning": "Model returned no response"}

    prompt = JUDGE_PROMPT.format(
        question=question,
        correct_answer=correct_answer,
        model_response=model_response,
    )

    delay = 4
    for attempt in range(max_retries):
        time.sleep(CALL_DELAY)
        raw    = call_model(JUDGE_MODEL, prompt, max_retries=3, timeout=45)
        parsed = extract_json(raw) if raw else None

        if parsed is not None:
            score = parsed.get("score")
            if score in (0, 1, 2):
                return {"score": int(score),
                        "reasoning": str(parsed.get("reasoning", ""))[:300]}
            print(f"  ⚠️  Judge gave invalid score '{score}' — retrying …")
        else:
            print(f"  ⚠️  Judge parse failed (attempt {attempt+1}). "
                  f"Raw: {repr(raw)[:120]}")

        if attempt < max_retries - 1:
            time.sleep(delay * (2 ** attempt))

    return {"score": -1, "reasoning": "Judge failed after all retries"}

# ── 6. Checkpoint helpers (atomic write) ──────────────────────────────────────
def ckpt_path(model_name):
    safe = re.sub(r"[/:\\]", "_", model_name)
    return os.path.join(CHECKPOINT_DIR, f"{safe}.json")

def load_checkpoint(model_name):
    path = ckpt_path(model_name)
    if os.path.exists(path):
        try:
            with open(path) as f:
                return json.load(f)
        except Exception as e:
            print(f"  ⚠️  Checkpoint corrupted ({e}) — starting fresh")
    return None

def save_checkpoint(model_name, results, last_index, total):
    path = ckpt_path(model_name)
    tmp  = path + ".tmp"
    try:
        with open(tmp, "w") as f:
            json.dump({"model": model_name, "last_index": last_index,
                       "results": results,
                       "saved_at": datetime.now().isoformat()}, f, indent=2)
        os.replace(tmp, path)
        print(f"  💾 Checkpoint saved at Q{last_index + 1}/{total}")
    except Exception as e:
        print(f"  ❌ Checkpoint save failed: {e}")

# ── 7. Load dataset ───────────────────────────────────────────────────────────
print("\n" + "═"*60)
print("STEP 1 — Loading dataset")
print("═"*60)

csvs = glob.glob("/content/*.csv")
if csvs:
    csv_path = csvs[0]
    print(f"📂 Found: {csv_path}")
else:
    from google.colab import files as _colab_files
    print("📤 No CSV in /content — upload it now:")
    _up = _colab_files.upload()
    csv_path = list(_up.keys())[0]

df = pd.read_csv(csv_path)
required = {"Question", "Answer", "Category"}
assert required.issubset(df.columns), f"Missing columns: {required - set(df.columns)}"
df = df.dropna(subset=list(required)).reset_index(drop=True)
print(f"✅ {len(df)} questions loaded")
print(df["Category"].value_counts().to_string())

# ── 8. Prompt template ────────────────────────────────────────────────────────
MODEL_PROMPT = (
    "You are a knowledgeable assistant specialising in Nigerian livestock "
    "management, ethnoveterinary practice, and tropical animal health. "
    "Answer the following question concisely and accurately.\n\n"
    "Question: {question}"
)

# ── 9. Pilot test (10 questions per model) ────────────────────────────────────
print("\n" + "═"*60)
print("STEP 2 — Pilot test (10 questions per model)")
print("═"*60)

def run_pilot(model, df, n=10):
    print(f"\n{'─'*55}\n🧪 PILOT — {model}\n{'─'*55}")

    cats    = df["Category"].unique()

    # FIX 1: Ensure we pull enough samples per category to reach exactly 'n'
    per_cat = max(1, (n // len(cats)) + 1)

    # FIX 2: Use a list comprehension instead of .groupby().apply() to avoid Pandas warnings
    frames = [df[df["Category"] == c].sample(n=min(len(df[df["Category"] == c]), per_cat), random_state=42) for c in cats]
    sample = pd.concat(frames).sample(frac=1, random_state=42).head(n).reset_index(drop=True)

    model_ok_n = judge_ok_n = 0
    actual_n   = len(sample) # This will now be exactly 10

    for i, row in sample.iterrows():
        q, a, cat = str(row["Question"]), str(row["Answer"]), str(row["Category"])

        time.sleep(CALL_DELAY)
        response = call_model(model, MODEL_PROMPT.format(question=q))
        m_ok     = response is not None

        verdict  = judge_response(q, a, response)   # has its own sleep
        j_ok     = verdict["score"] in (0, 1, 2)

        if m_ok:          model_ok_n += 1
        if m_ok and j_ok: judge_ok_n += 1

        icon = "✅" if (m_ok and j_ok) else ("⚠️ " if m_ok else "❌")
        print(f"  {icon} Q{i+1} [{cat[:24]}]  "
              f"model={'OK' if m_ok else 'FAIL'}  "
              f"judge={'OK('+str(verdict['score'])+')' if j_ok else 'FAIL'}")

    # Dynamically check for 80% pass rate
    pass_thresh = max(1, int(actual_n * 0.8))
    m_pass = model_ok_n >= pass_thresh
    j_pass = judge_ok_n >= pass_thresh

    print(f"\n  Model : {model_ok_n}/{actual_n} → {'✅ PASS' if m_pass else '❌ FAIL'}")
    print(f"  Judge : {judge_ok_n}/{actual_n} → {'✅ PASS' if j_pass else '❌ FAIL'}")
    return {"model_ok": m_pass, "judge_ok": j_pass}


pilot_results = {}
for model in MODELS_TO_EVALUATE:
    pilot_results[model] = run_pilot(model, df)

print(f"\n{'═'*60}\nPILOT SUMMARY — GO / NO-GO\n{'═'*60}")
approved_models = []
for model, res in pilot_results.items():
    go  = res["model_ok"] and res["judge_ok"]
    tag = "🟢 GO  " if go else "🔴 SKIP"
    print(f"  {tag} | {model}")
    if go:
        approved_models.append(model)

print(f"\n{len(approved_models)}/{len(MODELS_TO_EVALUATE)} model(s) approved")
if not approved_models:
    raise SystemExit("❌ No models passed the pilot. Check your API key and model names.")

# ── 10. Full evaluation ───────────────────────────────────────────────────────
print("\n" + "═"*60)
print("STEP 3 — Full evaluation")
print("═"*60)

def evaluate_model_full(model, df):
    print(f"\n{'═'*60}\n🔬 EVALUATING: {model}\n{'═'*60}")

    ckpt = load_checkpoint(model)
    if ckpt and ckpt.get("results"):
        results   = ckpt["results"]
        start_idx = ckpt["last_index"] + 1
        print(f"  📂 Resuming from Q{start_idx + 1}  ({len(results)} already done)")
    else:
        results, start_idx = [], 0
        print("  🆕 Starting fresh")

    if start_idx >= len(df):
        print("  ✅ Already complete — loaded from checkpoint")
        return results

    questions = df.to_dict("records")
    pbar = tqdm(range(start_idx, len(questions)),
                desc=model.replace(":free", ""),
                total=len(questions) - start_idx,
                unit="q")

    for i in pbar:
        row = questions[i]
        q, a, cat = str(row["Question"]), str(row["Answer"]), str(row["Category"])

        time.sleep(CALL_DELAY)
        response = call_model(model, MODEL_PROMPT.format(question=q))

        verdict = judge_response(q, a, response)   # has its own sleep

        results.append({
            "q_index":        i,
            "category":       cat,
            "question":       q,
            "correct_answer": a,
            "model_response": response,
            "score":          verdict["score"],
            "reasoning":      verdict["reasoning"],
        })

        valid = [r for r in results if r["score"] >= 0]
        if valid:
            avg = sum(r["score"] for r in valid) / len(valid)
            pbar.set_postfix(avg=f"{avg:.2f}", done=len(valid))

        if (i + 1) % CHECKPOINT_EVERY == 0 or i == len(questions) - 1:
            save_checkpoint(model, results, i, len(questions))

    pbar.close()
    print(f"  ✅ Done — {len(results)} questions evaluated")
    return results


all_results = {}
for model in approved_models:
    all_results[model] = evaluate_model_full(model, df)

print("\n🎉 All evaluations complete!")

# ── 11. Results & export ──────────────────────────────────────────────────────
print("\n" + "═"*65)
print("STEP 4 — Results")
print("═"*65)

def summarise(model, results):
    valid  = [r for r in results if r["score"] in (0, 1, 2)]
    failed = [r for r in results if r["score"] == -1]
    if not valid:
        return None
    scores = [r for r in valid if r["score"] >= 0]
    n      = len(scores)
    if n == 0:
        return None
    avg    = sum(r["score"] for r in scores) / n
    cat_sc = defaultdict(list)
    for r in valid:
        cat_sc[r["category"]].append(r["score"])
    return {
        "model":       model,
        "n_total":     len(results),
        "n_valid":     n,
        "n_failed":    len(failed),
        "avg_score":   round(avg, 4),
        "pct_correct": round(sum(1 for s in scores if s["score"] == 2) / n * 100, 1),
        "pct_partial": round(sum(1 for s in scores if s["score"] == 1) / n * 100, 1),
        "pct_wrong":   round(sum(1 for s in scores if s["score"] == 0) / n * 100, 1),
        "cat_avgs":    {c: round(sum(s)/len(s), 4) for c, s in cat_sc.items()},
    }

table_rows = []
for model, results in all_results.items():
    s = summarise(model, results)
    if not s:
        print(f"⚠️  {model} — no valid results")
        continue
    print(f"\n  Model : {model}")
    print(f"  Scores: avg={s['avg_score']}/2.0  |  "
          f"✅ {s['pct_correct']}% correct  "
          f"⚠️  {s['pct_partial']}% partial  "
          f"❌ {s['pct_wrong']}% wrong  "
          f"🚫 {s['n_failed']} no-response")
    print("  Per category:")
    for cat, v in sorted(s["cat_avgs"].items()):
        bar = "█" * int(v * 10)
        print(f"    {cat:<35} {v:.3f}  {bar}")

    row = {"model": model, "avg_score": s["avg_score"],
           "pct_correct": s["pct_correct"], "pct_partial": s["pct_partial"],
           "pct_wrong": s["pct_wrong"], "n_failed": s["n_failed"]}
    row.update({f"cat_{k.replace(' ','_')}": v for k, v in s["cat_avgs"].items()})
    table_rows.append(row)

summary_df = pd.DataFrame(table_rows).sort_values("avg_score", ascending=False)
print(f"\n{'═'*65}\nRANKING TABLE\n{'═'*65}")
print(summary_df.to_string(index=False))

# Save CSVs
ts        = datetime.now().strftime("%Y%m%d_%H%M")
full_rows = [{"model": m, **r} for m, rs in all_results.items() for r in rs]
full_df   = pd.DataFrame(full_rows)
full_path    = f"/content/bias_eval_full_{ts}.csv"
summary_path = f"/content/bias_eval_summary_{ts}.csv"
full_df.to_csv(full_path, index=False)
summary_df.to_csv(summary_path, index=False)
print(f"\n✅ Saved:\n   {full_path}\n   {summary_path}")

try:
    from google.colab import files as _colab_files
    _colab_files.download(full_path)
    _colab_files.download(summary_path)
    print("⬇&nbsp; Downloads triggered")
except Exception:
    print("ℹ️  Find files in the 📁 panel on the left under /content/")

# ─────────────────────────────────────────────────────────────────────────────
# TO RESUME AFTER A CRASH:
# 1. Re-run this entire block (it will re-ask for your API key)
# 2. Upload your CSV when prompted
# 3. Upload your checkpoint .json files to /content/checkpoints/ via the
#    left-side file panel, then the script will auto-resume from where it stopped
# ─────────────────────────────────────────────────────────────────────────────


🔑 Paste your Groq API key (gsk_...) and press Enter: ··········
✅ Config ready  |  models: 2  |  judge: llama-3.1-8b-instant

════════════════════════════════════════════════════════════
STEP 1 — Loading dataset
════════════════════════════════════════════════════════════
📂 Found: /content/nigerian_livestock_standardized_70.csv
✅ 420 questions loaded
Category
Ethnoveterinary                 70
Local Treatment Context         70
Terminology                     70
Tropical Disease Knowledge      70
Breed Knowledge                 70
Production & General Context    70

════════════════════════════════════════════════════════════
STEP 2 — Pilot test (10 questions per model)
════════════════════════════════════════════════════════════

───────────────────────────────────────────────────────
🧪 PILOT — llama-3.1-8b-instant
───────────────────────────────────────────────────────
  ✅ Q1 [Production & General Con]  model=OK  judge=OK(0)
  ✅ Q2 [Breed Knowledge]  model=OK  judge=OK(1)
  ✅ Q3 [Eth

llama-3.1-8b-instant:   0%|          | 0/420 [00:00<?, ?q/s]

  💾 Checkpoint saved at Q70/420
  💾 Checkpoint saved at Q140/420
  💾 Checkpoint saved at Q210/420
  💾 Checkpoint saved at Q280/420
  💾 Checkpoint saved at Q350/420
  💾 Checkpoint saved at Q420/420
  ✅ Done — 420 questions evaluated

🎉 All evaluations complete!

═════════════════════════════════════════════════════════════════
STEP 4 — Results
═════════════════════════════════════════════════════════════════

  Model : llama-3.1-8b-instant
  Scores: avg=1.2952/2.0  |  ✅ 46.9% correct  ⚠️  35.7% partial  ❌ 17.4% wrong  🚫 0 no-response
  Per category:
    Breed Knowledge                     1.271  ████████████
    Ethnoveterinary                     1.186  ███████████
    Local Treatment Context             1.343  █████████████
    Production & General Context        1.357  █████████████
    Terminology                         1.171  ███████████
    Tropical Disease Knowledge          1.443  ██████████████

═════════════════════════════════════════════════════════════════
RANKING TABLE
══

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇&nbsp; Downloads triggered
